In [52]:
import requests
import os
import json
import pandas as pd
from dotenv import load_dotenv, dotenv_values
from requests.exceptions import HTTPError, ConnectionError, Timeout, RequestException
from pathlib import Path

In [67]:
load_dotenv(Path.cwd().parent / ".env")

True

In [68]:
URL_BASE = "https://api.company-information.service.gov.uk"
URL_META_DATA = "https://document-api.company-information.service.gov.uk/document"
TIMEOUT = 10  # seconds

In [69]:
#Company House

def get(endpoint, params=None, headers=None):
    url = f"{URL_BASE}/{endpoint}"
    response = requests.get(
        url,
        auth=(os.getenv("CH_API"),""),
        headers=headers,
        params=params,
        timeout=TIMEOUT
        )
    response.raise_for_status()
    print(response.status_code)
    return response.json()

def get_meta_data(endpoint, params=None, headers=None):
    url = f"{URL_META_DATA}/{endpoint}"
    response = requests.get(
        url,
        auth=(os.getenv("CH_API"),""),
        headers=headers,
        params=params,
        timeout=TIMEOUT
        )
    response.raise_for_status()
    return response

In [78]:
company_number = "00002065"

profile = get(f"/company/{company_number}")
profile

200


{'accounts': {'accounting_reference_date': {'day': '31', 'month': '12'},
  'last_accounts': {'made_up_to': '2025-12-31',
   'period_end_on': '2025-12-31',
   'period_start_on': '2025-01-01',
   'type': 'group'},
  'next_accounts': {'due_on': '2027-06-30',
   'overdue': False,
   'period_end_on': '2026-12-31',
   'period_start_on': '2026-01-01'},
  'next_due': '2027-06-30',
  'next_made_up_to': '2026-12-31',
  'overdue': False},
 'can_file': True,
 'company_name': 'LLOYDS BANK PLC',
 'company_number': '00002065',
 'company_status': 'active',
 'confirmation_statement': {'last_made_up_to': '2026-05-06',
  'next_due': '2027-05-20',
  'next_made_up_to': '2027-05-06',
  'overdue': False},
 'date_of_creation': '1865-04-20',
 'etag': '8259055527962f7d8a7133c1d55c10b900e50b46',
 'has_charges': False,
 'has_insolvency_history': False,
 'jurisdiction': 'england-wales',
 'last_full_members_list_date': '2016-05-09',
 'links': {'persons_with_significant_control': '/company/00002065/persons-with-sign

In [75]:
profile["company_number"]

'12247029'

In [76]:
# Filling History
data = get(f"/company/{company_number}/filing-history")

for i in data["items"]:
    print(i)

200
{'transaction_id': 'MzMxNjgwODI3M2FkaXF6a2N4', 'type': 'GAZ2(A)', 'date': '2021-10-19', 'category': 'gazette', 'description': 'gazette-dissolved-voluntary', 'pages': 1, 'paper_filed': True, 'links': {'self': '/company/12247029/filing-history/MzMxNjgwODI3M2FkaXF6a2N4', 'document_metadata': 'https://document-api.company-information.service.gov.uk/document/hfrP57D4zw2WcPApqsHv14z-YaNiNrUz3pKeiy2E6Qw'}}
{'transaction_id': 'MzMwODU4NjY3MWFkaXF6a2N4', 'type': 'GAZ1(A)', 'date': '2021-08-03', 'category': 'gazette', 'description': 'gazette-notice-voluntary', 'pages': 1, 'paper_filed': True, 'links': {'self': '/company/12247029/filing-history/MzMwODU4NjY3MWFkaXF6a2N4', 'document_metadata': 'https://document-api.company-information.service.gov.uk/document/mAO6sDGWhHbnCoYUjZsnhPqHkYvSHgCbDVYuDfoX1Vg'}}
{'transaction_id': 'MzMwODU4NjAwM2FkaXF6a2N4', 'barcode': 'XA9DE1BS', 'type': 'DS01', 'date': '2021-07-23', 'category': 'dissolution', 'description': 'dissolution-application-strike-off-company

In [77]:
x = pd.DataFrame.from_dict(data["items"])

In [59]:
x = x[["transaction_id","barcode","type","date","category","subcategory","paper_filed","links"]]

In [60]:
# Charges
data = get(f"/company/{company_number}/charges", params={"items_per_page": 100, "start_index":0})
data

200


{'etag': '54a907a708ff44e333154169a9d85512c06417cc',
 'total_count': 87,
 'unfiltered_count': 87,
 'satisfied_count': 18,
 'part_satisfied_count': 0,
 'items': [{'etag': '51fe069c10b6eeb0ff2ae5421887ea568e9d0990',
   'charge_code': '000020650090',
   'classification': {'type': 'charge-description',
    'description': 'A registered charge'},
   'charge_number': 90,
   'status': 'outstanding',
   'delivered_on': '2026-05-08',
   'created_on': '2026-05-06',
   'particulars': {'type': 'brief-description',
    'description': 'N/A.',
    'contains_fixed_charge': True},
   'more_than_four_persons_entitled': True,
   'persons_entitled': [{'name': "The Society Incorporated by Lloyd's Act 1871 by the Name of Lloyd's"},
    {'name': 'The Trustees (As Defined in the Instrument)'},
    {'name': 'The Beneficiaries (As Defined in the Instrument)'},
    {'name': 'The Premiums Trustees (As Defined in the Instrument)'}],
   'transactions': [{'filing_type': 'create-charge-with-deed',
     'delivered_on':

In [61]:
x = pd.DataFrame.from_dict(data["items"])

In [62]:
x

,etag,charge_code,classification,charge_number,status,delivered_on,created_on,particulars,more_than_four_persons_entitled,persons_entitled,transactions,links,satisfied_on,assets_ceased_released,secured_details
0,51fe069c10b6eeb0ff2ae5421887ea568e9d0990,000020650090,"{'type': 'charge-description', 'description': ...",90,outstanding,2026-05-08,2026-05-06,"{'type': 'brief-description', 'description': '...",True,[{'name': 'The Society Incorporated by Lloyd's...,"[{'filing_type': 'create-charge-with-deed', 'd...",{'self': '/company/00002065/charges/k7RobJw5L_...,NaN,NaN,NaN
1,a7e0c4d698c20ed5454b25587bbf5c3fbf908fb0,000020650089,"{'type': 'charge-description', 'description': ...",89,outstanding,2025-06-30,2025-06-23,"{'type': 'brief-description', 'description': '...",True,[{'name': 'The Society Incorporated by Lloyd's...,"[{'filing_type': 'create-charge-with-deed', 'd...",{'self': '/company/00002065/charges/-5jHzmgIkf...,NaN,NaN,NaN
2,6e8315b09b34d959a6065ec9cb798df48c0c6f5f,000020650088,"{'type': 'charge-description', 'description': ...",88,outstanding,2025-03-18,2025-03-18,"{'type': 'brief-description', 'description': '...",True,[{'name': 'The Society Incorporated by Lloyd's...,"[{'filing_type': 'create-charge-with-deed', 'd...",{'self': '/company/00002065/charges/t5EdSnhz-x...,NaN,NaN,NaN
3,8b98dba1da4c4fa9fd859b8554e31f4432bd9b35,000020650087,"{'type': 'charge-description', 'description': ...",87,outstanding,2025-03-17,2025-03-11,"{'contains_fixed_charge': True, 'contains_nega...",NaN,[{'name': 'Kfw as Assignee'}],"[{'filing_type': 'create-charge-with-deed', 'd...",{'self': '/company/00002065/charges/w7t6sAi91v...,NaN,NaN,NaN
4,0d751a48fa558521449c14cee3adf28a6417430b,000020650086,"{'type': 'charge-description', 'description': ...",86,outstanding,2025-03-17,2025-03-11,"{'contains_fixed_charge': True, 'contains_nega...",NaN,[{'name': 'Kfw as Assignee'}],"[{'filing_type': 'create-charge-with-deed', 'd...",{'self': '/company/00002065/charges/TP76wvyn09...,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,NaN,NaN,"{'type': 'charge-description', 'description': ...",5,fully-satisfied,1997-08-20,1997-08-04,"{'type': 'short-particulars', 'description': '...",NaN,[{'name': 'Federal Reserve Bank of Atlanta'}],[{'filing_type': 'create-charge-pre-2006-compa...,{'self': '/company/00002065/charges/BVuTiVO3og...,1998-02-23,NaN,"{'type': 'amount-secured', 'description': 'All..."
83,NaN,NaN,"{'type': 'charge-description', 'description': ...",3,fully-satisfied,1992-12-15,1992-11-26,"{'type': 'short-particulars', 'description': '...",NaN,[{'name': 'The Governor and Company of the Ban...,[{'filing_type': 'create-charge-pre-2006-compa...,{'self': '/company/00002065/charges/qhRzwPzbyy...,2000-03-06,NaN,"{'type': 'amount-secured', 'description': 'All..."
84,NaN,NaN,"{'type': 'charge-description', 'description': ...",4,fully-satisfied,1985-07-26,1985-07-15,"{'type': 'short-particulars', 'description': '...",NaN,[{'name': 'Lloyds Bank PLC'}],"[{'filing_type': 'create-charge', 'delivered_o...",{'self': '/company/00002065/charges/qvoAk8xMDE...,1993-06-17,NaN,"{'type': 'amount-secured', 'description': 'All..."
85,NaN,NaN,"{'type': 'charge-description', 'description': ...",2,fully-satisfied,1984-07-10,1984-06-22,"{'type': 'short-particulars', 'description': '...",NaN,[{'name': 'Schroder Life Assurance Limited'}],"[{'filing_type': 'create-charge', 'delivered_o...",{'self': '/company/00002065/charges/O0brOUWmIn...,1984-12-05,NaN,"{'type': 'amount-secured', 'description': 'All..."


In [63]:
data = get(f"/company/{company_number}/persons-with-significant-control")
data

# x = pd.DataFrame.from_dict(data)
# x

200


{'items_per_page': 25,
 'items': [{'etag': '0110718503ce0622dd453f7f01c75cc97c3b80c8',
   'notified_on': '2016-04-06',
   'name': 'Lloyds Banking Group Plc',
   'links': {'self': '/company/00002065/persons-with-significant-control/corporate-entity/rTHhnY4-WO4nqU_grhl-RUEB6z0'},
   'identification': {'legal_form': 'Public Limited Company',
    'legal_authority': 'United Kingdom (Scotland)',
    'country_registered': 'Scotland',
    'place_registered': 'Companies House',
    'registration_number': 'Sc095000'},
   'ceased': False,
   'kind': 'corporate-entity-person-with-significant-control',
   'address': {'address_line_1': 'The Mound',
    'country': 'United Kingdom',
    'locality': 'Edinburgh',
    'postal_code': 'EH1 1YZ'},
   'natures_of_control': ['ownership-of-shares-75-to-100-percent',
    'voting-rights-75-to-100-percent',
    'right-to-appoint-and-remove-directors']}],
 'start_index': 0,
 'total_results': 1,
 'active_count': 1,
 'ceased_count': 0,
 'links': {'self': '/company/0

In [64]:
data = get(f"/company/{company_number}/officers")

temp = data["items"][0]["links"]["self"].split("/")

200


In [65]:
x = pd.DataFrame.from_dict(data["items"])
x[10:15]

,etag,address,appointed_on,is_pre_1992_appointment,links,name,officer_role,person_number,country_of_residence,date_of_birth,nationality,former_names,identity_verification_details,resigned_on,appointed_before
10,cf24bc3fa139ab0d57532bd822f36cbfbc18f463,"{'address_line_1': 'Gresham Street', 'country'...",2021-08-16,False,{'self': '/company/00002065/appointments/VIRkO...,"NUNN, Charles Alan",director,286317400001,United Kingdom,"{'month': 9, 'year': 1971}",British,NaN,{'anti_money_laundering_supervisory_bodies': [...,NaN,NaN
11,a4d7a5e57dc6cff43fc17dbc39798848c999bffc,"{'address_line_1': 'Gresham Street', 'country'...",2022-11-01,False,{'self': '/company/00002065/appointments/ZKO8d...,"TURNER, Catherine Lucy",director,180221240001,United Kingdom,"{'month': 6, 'year': 1963}",British,NaN,{'anti_money_laundering_supervisory_bodies': [...,NaN,NaN
12,d91ebca67f940381c3fb74ca41876cb546c22f9e,"{'address_line_1': 'Old Broad Street', 'countr...",2025-06-16,False,{'self': '/company/00002065/appointments/nguSG...,"VOGELZANG, Christiaan Franciscus Henricus Herman",director,336954640001,Netherlands,"{'month': 11, 'year': 1962}",Dutch,NaN,{'anti_money_laundering_supervisory_bodies': [...,NaN,NaN
13,c8cdf79e126d92dcccbdaf34954c6b5547267759,"{'address_line_1': 'Gresham Street', 'country'...",2020-03-01,False,{'self': '/company/00002065/appointments/rmuDT...,"WOODS, Catherine Marie",director,268099680001,Ireland,"{'month': 9, 'year': 1962}",Irish,"[{'forenames': 'CATHERINE', 'surname': 'MARIE ...",{'anti_money_laundering_supervisory_bodies': [...,NaN,NaN
14,3bdaa1003f7b020a8dc53ce11bf5010b30a0f93d,"{'address_line_1': 'The Mound', 'locality': 'E...",2009-01-16,False,{'self': '/company/00002065/appointments/j0HG6...,"BAINES, Harold Francis",secretary,207382860001,NaN,NaN,British,NaN,NaN,2012-08-01,NaN


In [66]:
data = get(f"/company/{company_number}/officers")
data

200


{'active_count': 14,
 'etag': '3bdaa1003f7b020a8dc53ce11bf5010b30a0f93d',
 'items': [{'etag': '3bdaa1003f7b020a8dc53ce11bf5010b30a0f93d',
   'address': {'address_line_1': 'Gresham Street',
    'country': 'United Kingdom',
    'locality': 'London',
    'postal_code': 'EC2V 7HN',
    'premises': '25'},
   'appointed_on': '2019-07-01',
   'is_pre_1992_appointment': False,
   'links': {'self': '/company/00002065/appointments/s69oD4uF8aj7KGtzyBtyTyPD-2Q',
    'officer': {'appointments': '/officers/xeS7C_yO94cCW7iWxHLPA5thIaw/appointments'}},
   'name': 'CHEETHAM, Catharine Lucy',
   'officer_role': 'secretary',
   'person_number': '260163850001'},
  {'etag': 'a4694fa692a5dc6636fa40a6ed90a8f194ad2fb4',
   'address': {'address_line_1': 'Gresham Street',
    'country': 'United Kingdom',
    'locality': 'London',
    'postal_code': 'EC2V 7HN',
    'premises': '25'},
   'appointed_on': '2019-01-01',
   'is_pre_1992_appointment': False,
   'country_of_residence': 'United Kingdom',
   'date_of_bir

In [73]:
# Useable APIs
#   - Charges - number of fully satisfied / outstanding charges, timeline, borrowing freqeuncy
#   - Officer - prominent figures in the company (tracked through person_number), likely can be used to 
#       track their history through different companies. In the companies themselves, we can see the 
#       number of resignations (resignation date)
#   - Person of Significant Influence - share holders, likely can use the same mechanisms as officers
#   - Account - only thing is probably the has_charges, history_of_insolvancy, next_accounts, 
#       sic_code(industry code) 

# All the companies can be tracked through their company numbers, which hopefully will be available in all 
# API channels